# 20：19の仮想データPKLを選択し、モデル・元データとの整合性を検証
19で出力したPKLを番号で選択します。モデルの重みだけでは元データの分布は確認できないため、19の`provenance.json`からモデル・前処理情報を特定し、18に保存した学習用CSV・ホールドアウトCSVと比較します。

主な検証：モデルSHA256、件数・列・型・カテゴリー、欠損率、数値分布、カテゴリー割合、数値相関、日時・経過時間・処置の論理的不一致。

元データや生成データを変更せず、結果を`artifacts/20_tabdiff_validation/実行時刻/`に保存します。再学習や再生成は行いません。
統計的な差が小さいことだけで臨床的妥当性・匿名性を保証するものではありません。
修正版：生成直後の欠損フラグ、独立復元、19の復元保存表、後処理後を比較します。通常/EMAの同じ学習実行の欠損率比較も保存します。


In [ ]:
from pathlib import Path
from datetime import datetime
import json, hashlib, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import ks_2samp, wasserstein_distance
from IPython.display import display

PROJECT = Path.cwd().resolve()
if PROJECT.name == "notebooks":
    PROJECT = PROJECT.parent
GENERATED_ROOT = PROJECT / "artifacts/19_tabdiff_generate"
TABDIFF_REPO = PROJECT / "external/TabDiff"
SEED = 42
MAX_NUMERIC_SAMPLE = 100_000  # 分布指標の計算上限。カテゴリー・欠損率は全件集計
MAX_CORRELATION_ROWS = 50_000
# 下記は「要確認」の運用目安であり、統計学的に保証された合格基準ではありません。
MISSING_DIFF_LIMIT = 0.05
KS_LIMIT = 0.10
SMD_LIMIT = 0.20
TV_LIMIT = 0.10
CORRELATION_DIFF_LIMIT = 0.15
assert GENERATED_ROOT.is_dir(), f"19の保存先がありません: {GENERATED_ROOT}"

In [ ]:
# 複数の生成PKLを一覧にします。ログ・前処理PKLは対象外。
files = sorted(GENERATED_ROOT.glob("*/utstein_synthetic_*.pkl"))
if not files:
    raise FileNotFoundError(f"生成PKLがありません: {GENERATED_ROOT}")
rows = []
for i, path in enumerate(files):
    manifest = path.parent / "provenance.json"
    try:
        p = json.loads(manifest.read_text(encoding="utf-8")) if manifest.is_file() else {}
        problem = "" if p else "モデル由来情報なし"
    except (ValueError, OSError):
        p, problem = {}, "由来情報の読込不可"
    rows.append({"番号": i, "生成実行": path.parent.name, "ファイル": path.name,
                 "予定件数": p.get("n_generate"),
                 "使用モデル": str(p.get("checkpoint", "不明")).split("/")[-1],
                 "種別": p.get("model_type", "旧版/不明"), "EMA更新": p.get("ema_update", "旧版/不明"),
                 "状態": problem or "由来情報あり", "パス": str(path)})
file_table = pd.DataFrame(rows)
display(file_table.drop(columns="パス"))

In [ ]:
FILE_NUMBER = 0  # ← 上の番号から検証したいPKLを選択

if FILE_NUMBER not in file_table["番号"].values:
    raise ValueError("一覧にある番号を指定してください。")
SYNTHETIC_PATH = Path(file_table.loc[file_table["番号"].eq(FILE_NUMBER), "パス"].iloc[0])
GEN_DIR = SYNTHETIC_PATH.parent
provenance = json.loads((GEN_DIR / "provenance.json").read_text(encoding="utf-8"))
generation_config = json.loads((GEN_DIR / "run_config.json").read_text(encoding="utf-8"))
name = generation_config["dataname"]
exp = generation_config["exp_name"]
if any(Path(x).name != x for x in [name, exp]):
    raise ValueError("設定のデータ名・実験名が不正です。")
# 移動前の絶対パスがあっても、現在のプロジェクト内を優先して解決します。
source_local = PROJECT / "artifacts/18_tabdiff" / Path(provenance["source_run"]).name
SOURCE_RUN = source_local if source_local.is_dir() else Path(provenance["source_run"])
PREPROCESS_PATH = SOURCE_RUN / "preprocessing.pkl"
CHECKPOINT = TABDIFF_REPO / "tabdiff/ckpt" / name / exp / Path(provenance["checkpoint"]).name
if not CHECKPOINT.is_file():
    CHECKPOINT = Path(provenance["checkpoint"])
DATA_DIR = TABDIFF_REPO / "data" / name
TRAIN_CSV = DATA_DIR / f"{name}.csv"  # 18が保存した、補完＋欠損フラグ形式の学習用表
HOLDOUT_CSV = DATA_DIR / "holdout.csv"
for path in [PREPROCESS_PATH, CHECKPOINT, CHECKPOINT.parent / "config.pkl",
             SOURCE_RUN / "run_config.json", TRAIN_CSV, HOLDOUT_CSV, DATA_DIR / "info.json"]:
    if not path.is_file():
        raise FileNotFoundError(f"検証に必要なファイルがありません: {path}")
source_config = json.loads((SOURCE_RUN / "run_config.json").read_text(encoding="utf-8"))
if source_config["dataname"] != name or source_config["exp_name"] != exp:
    raise ValueError("19と18の学習実行設定が一致しません。")
preprocessing = pd.read_pickle(PREPROCESS_PATH)
schema, constants = preprocessing["schema"], preprocessing["constants"]
expected_columns = preprocessing["columns"]
synthetic = pd.read_pickle(SYNTHETIC_PATH)
if not isinstance(synthetic, pd.DataFrame) or not synthetic.columns.is_unique:
    raise ValueError("PKLは列名が一意のDataFrameである必要があります。")
OUT = PROJECT / "artifacts/20_tabdiff_validation" / datetime.now().strftime("%Y%m%d_%H%M%S_%f")
OUT.mkdir(parents=True, exist_ok=False)
print("検証対象:", SYNTHETIC_PATH)
print("モデル:", CHECKPOINT)
print("検証結果:", OUT)

In [ ]:
# モデル重みとメタデータの対応を確認します。重みを変更・実行しません。
def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(1024**2), b""):
            h.update(block)
    return h.hexdigest()

expected_keys = set()
for spec in schema.values():
    expected_keys.add(spec["key"])
    if "mask" in spec:
        expected_keys.add(spec["mask"])
info = json.loads((DATA_DIR / "info.json").read_text(encoding="utf-8"))
actual_model_hash = sha256_file(CHECKPOINT)
expected_hash = provenance.get("checkpoint_sha256")
metadata_checks = [
    {"項目": "モデル重みのSHA256", "結果": "一致" if expected_hash == actual_model_hash else "不一致または記録なし"},
    {"項目": "前処理とモデル入力列", "結果": "一致" if expected_keys == set(info["column_names"]) else "不一致"},
    {"項目": "生成件数", "結果": "一致" if len(synthetic) == provenance.get("n_generate") else "不一致"},
    {"項目": "出力列と列順", "結果": "一致" if synthetic.columns.tolist() == expected_columns else "不一致"},
]
metadata_report = pd.DataFrame(metadata_checks)
display(metadata_report)
metadata_report.to_csv(OUT / "model_metadata_checks.csv", index=False, encoding="utf-8-sig")
if not expected_hash or actual_model_hash != expected_hash:
    raise ValueError("19で記録したモデルのSHA256と一致しません。別の重みが混在していないか確認してください。")
if expected_keys != set(info["column_names"]):
    raise ValueError("前処理情報とモデルの入力列が不一致です。")
# 新版が記録した各資産・生成表のハッシュも検証。旧版は未記録と報告します。
asset_rows = []
assets = provenance.get("training_assets", {})
for label, path, expected in [
    ("前処理", PREPROCESS_PATH, assets.get("preprocessing_sha256")),
    ("学習CSV", TRAIN_CSV, assets.get("train_csv_sha256")),
    ("ホールドアウトCSV", HOLDOUT_CSV, assets.get("holdout_csv_sha256")),
    ("選択PKL", SYNTHETIC_PATH, provenance.get("synthetic_sha256")),
    ("生成内部表", GEN_DIR / "generated_encoded.csv", provenance.get("generated_encoded_sha256")),
    ("後処理前PKL", GEN_DIR / "generated_restored_before_postprocessing.pkl", provenance.get("restored_before_postprocessing_sha256")),
]:
    status = "未記録（旧版）" if not expected else ("一致" if path.is_file() and sha256_file(path) == expected else "不一致/ファイルなし")
    asset_rows.append({"項目":label,"結果":status})
asset_report = pd.DataFrame(asset_rows)
display(asset_report)
asset_report.to_csv(OUT / "asset_hash_checks.csv", index=False, encoding="utf-8-sig")
if asset_report["結果"].eq("不一致/ファイルなし").any():
    raise ValueError("記録された資産が変更されています。検証対象を確認してください。")


## 学習・ホールドアウト表の復元
18が保存した表の補完値と欠損フラグから、元の欠損を戻します。比較対象には生成データ向けのクリップ・日付再構成を適用しません。
旧18/19は元CSV・前処理ファイル自体のハッシュを記録していないため（新版は追加検証）、同じ内容が保たれていることまで暗号学的には確認できません。モデルのハッシュと、列・件数・カテゴリーの対応で検証します。

In [ ]:
def decode_reference(encoded):
    missing = expected_keys - set(encoded.columns)
    if missing:
        raise ValueError(f"比較表に必要な内部列が不足: {sorted(missing)}")
    decoded = pd.DataFrame(index=encoded.index)
    for column, spec in schema.items():
        values = encoded[spec["key"]]
        if spec["kind"] == "category":
            values = values.astype(str)
            bad = ~values.isin(list(spec["decode"]) + ["missing_category"])
            if bad.any():
                raise ValueError(f"比較表に未定義カテゴリー: {column}")
            values = values.map(spec["decode"])
            if isinstance(spec["dtype"], pd.CategoricalDtype):
                mapping = {str(v): v for v in spec["dtype"].categories}
                values = pd.Series(pd.Categorical(values.map(mapping), dtype=spec["dtype"]), index=encoded.index)
            decoded[column] = values
        else:
            values = pd.to_numeric(values, errors="raise").astype(float)
            if "mask" in spec:
                mask_values = encoded[spec["mask"]].astype(str)
                if not mask_values.isin(["missing_yes", "missing_no"]).all():
                    raise ValueError(f"比較表に不正な欠損フラグ: {column}")
                values = values.mask(mask_values.eq("missing_yes"))
            if spec["kind"] == "datetime":
                decoded[column] = pd.Timestamp("1970-01-01") + pd.to_timedelta(values, unit="D")
            else:
                decoded[column] = values
    for column, value in constants.items():
        decoded[column] = value
    return decoded[expected_columns]

train_encoded = pd.read_csv(TRAIN_CSV, keep_default_na=False)
holdout_encoded = pd.read_csv(HOLDOUT_CSV, keep_default_na=False)
train = decode_reference(train_encoded)
holdout = decode_reference(holdout_encoded)
del train_encoded, holdout_encoded
if len(train) != len(preprocessing["train_idx"]) or len(holdout) != len(preprocessing["test_idx"]):
    raise ValueError("学習・ホールドアウト件数が保存済み前処理情報と一致しません。")
if len(synthetic) == 0:
    raise ValueError("生成データが0件です。")
references = {"学習": train, "ホールドアウト": holdout}
print(f"学習 {len(train):,}件 / ホールドアウト {len(holdout):,}件 / 生成 {len(synthetic):,}件")

In [ ]:
# 欠けた列・余分な列は報告し、比較可能な列を検証します。
common = [c for c in expected_columns if c in synthetic]
structural_rows = []
for column in list(dict.fromkeys(expected_columns + synthetic.columns.tolist())):
    if column not in synthetic:
        structural_rows.append({"列名": column, "結果": "出力列なし"}); continue
    if column not in expected_columns:
        structural_rows.append({"列名": column, "結果": "想定外の列"}); continue
    spec = schema.get(column)
    s = synthetic[column]
    if spec is None:
        val = constants[column]
        violation = ~s.isna() if pd.isna(val) else ~s.eq(val).fillna(False)
        unknown = int(violation.sum())
        status = "定数一致" if unknown == 0 else "定数不一致"
        expected_dtype = "定数（旧前処理では型未保存）"
    else:
        expected_dtype = str(spec["dtype"])
        if spec["kind"] == "category":
            unknown = int((s.notna() & ~s.astype("string").isin(list(spec["decode"].values()))).sum())
            if isinstance(spec["dtype"], pd.CategoricalDtype):
                compatible = isinstance(s.dtype, pd.CategoricalDtype) and s.dtype == spec["dtype"]
            else:
                compatible = not pd.api.types.is_numeric_dtype(s) and not pd.api.types.is_datetime64_any_dtype(s)
        elif spec["kind"] == "datetime":
            unknown = int((s.notna() & pd.to_datetime(s, errors="coerce").isna()).sum())
            compatible = pd.api.types.is_datetime64_any_dtype(s)
        else:
            values = pd.to_numeric(s, errors="coerce")
            unknown = int((s.notna() & (values.isna() | ~np.isfinite(values.astype(float)))).sum())
            compatible = pd.api.types.is_numeric_dtype(s)
        status = "適合" if unknown == 0 and compatible else "要確認"
    structural_rows.append({"列名": column, "結果": status, "想定型": expected_dtype,
        "生成型": str(s.dtype), "不正値件数": unknown})
structure_report = pd.DataFrame(structural_rows)
display(structure_report)
structure_report.to_csv(OUT / "schema_checks.csv", index=False, encoding="utf-8-sig")

In [ ]:
# 欠損パターンが崩れた段階を調べます。旧19の内部CSVにも対応します。
raw_path = GEN_DIR / "generated_encoded.csv"
stage_rows = []
if raw_path.is_file():
    raw = pd.read_csv(raw_path, keep_default_na=False)
    if len(raw) != len(synthetic):
        raise ValueError("内部表と選択PKLの件数が異なります。")
    raw_restored = decode_reference(raw)
    before_path = GEN_DIR / "generated_restored_before_postprocessing.pkl"
    saved_before = pd.read_pickle(before_path) if before_path.is_file() else None
    for column, spec in schema.items():
        if column not in synthetic:
            continue
        if spec["kind"] == "category":
            values = raw[spec["key"]].astype(str)
            unknown = ~values.isin(list(spec["decode"]) + ["missing_category"])
            if unknown.any():
                raise ValueError(f"不正な生成カテゴリー: {column}")
            mask = values.eq("missing_category")
        elif "mask" in spec:
            values = raw[spec["mask"]].astype(str)
            if not values.isin(["missing_yes", "missing_no"]).all():
                raise ValueError(f"不正な欠損フラグ: {column}")
            mask = values.eq("missing_yes")
        else:
            mask = pd.Series(False, index=raw.index)
        restored_mask = raw_restored[column].isna().reset_index(drop=True)
        final_mask = synthetic[column].isna().reset_index(drop=True)
        if saved_before is not None:
            if len(saved_before)!=len(raw) or column not in saved_before:
                raise ValueError("後処理前表の構造が一致しません。")
            saved_mask = saved_before[column].isna().reset_index(drop=True)
            decoder_mismatch = int(restored_mask.ne(saved_mask).sum())
        else:
            decoder_mismatch = None
        holdout_rate = float(holdout[column].isna().mean())
        raw_rate = float(mask.mean())
        stage_rows.append({"列名":column,"学習欠損率":float(train[column].isna().mean()),
            "ホールドアウト欠損率":holdout_rate,"生成フラグ欠損率":raw_rate,
            "独立復元欠損率":float(restored_mask.mean()),"最終欠損率":float(final_mask.mean()),
            "フラグ対独立復元の不一致件数":int(mask.ne(restored_mask).sum()),
            "19保存復元表との不一致件数":decoder_mismatch,
            "後処理で増えた欠損件数":int((~restored_mask & final_mask).sum()),
            "後処理で解消した欠損件数":int((restored_mask & ~final_mask).sum()),
            "生成時点で欠損率差大":abs(raw_rate-holdout_rate)>MISSING_DIFF_LIMIT})
    stage_report = pd.DataFrame(stage_rows)
    stage_report["生成時点の絶対差"] = (stage_report["生成フラグ欠損率"]-stage_report["ホールドアウト欠損率"]).abs()
    display(stage_report.sort_values("生成時点の絶対差",ascending=False).head(30))
    stage_report.to_csv(OUT / "missingness_stage_diagnosis.csv",index=False,encoding="utf-8-sig")
    del raw, raw_restored
else:
    stage_report = pd.DataFrame()
    print("generated_encoded.csvがないため欠損の段階別診断は省略しました。")


In [ ]:
# 欠損率は全件比較。差は割合（0.05 = 5パーセントポイント）。
missing_rows = []
for label, reference in references.items():
    for column in common:
        a, b = float(reference[column].isna().mean()), float(synthetic[column].isna().mean())
        missing_rows.append({"比較対象": label, "列名": column, "元データ欠損率": a,
            "生成欠損率": b, "絶対差": abs(a-b), "要確認": abs(a-b) > MISSING_DIFF_LIMIT})
missing_report = pd.DataFrame(missing_rows)
display(missing_report.sort_values("絶対差", ascending=False).head(30))
missing_report.to_csv(OUT / "missingness_comparison.csv", index=False, encoding="utf-8-sig")

## 分布比較
- 数値：平均・SD・分位点、KS距離、標準化平均差（SMD）、Wasserstein距離。日時は1970年からの日数で比較します。
- カテゴリー：各値の割合と総変動距離（TV）。欠損は別途比較するため、分布は非欠損値が対象です。
- 大規模データのp値だけで判断せず、差の大きさを表示します。運用目安超過は「要確認」であり、誤りの確定ではありません。

In [ ]:
def numeric_values(series, kind="numeric"):
    if kind == "datetime":
        values = (pd.to_datetime(series, errors="coerce") - pd.Timestamp("1970-01-01")).dt.total_seconds()/86400
    else:
        values = pd.to_numeric(series, errors="coerce")
    return values.replace([np.inf, -np.inf], np.nan).dropna().astype(float)

def sampled(values):
    return values.sample(n=min(MAX_NUMERIC_SAMPLE, len(values)), random_state=SEED).to_numpy()

numeric_rows, category_rows, proportion_rows = [], [], []
for column in common:
    spec = schema.get(column)
    if spec is None:
        continue
    if spec["kind"] in ["numeric", "datetime"]:
        syn = numeric_values(synthetic[column], spec["kind"])
        for label, reference in references.items():
            ref = numeric_values(reference[column], spec["kind"])
            row = {"比較対象": label, "列名": column, "元有効件数": len(ref), "生成有効件数": len(syn)}
            if len(ref) and len(syn):
                a, b = sampled(ref), sampled(syn)
                ks = float(ks_2samp(a, b).statistic)
                pooled = np.sqrt((ref.var(ddof=0) + syn.var(ddof=0))/2)
                smd = float(abs(syn.mean()-ref.mean())/pooled) if pooled else (0. if syn.mean()==ref.mean() else float("inf"))
                row.update(KS=ks, SMD=smd, Wasserstein=float(wasserstein_distance(a,b)),
                    元平均=ref.mean(), 生成平均=syn.mean(), 元SD=ref.std(), 生成SD=syn.std(),
                    元中央値=ref.median(), 生成中央値=syn.median(),
                    元1パーセント点=ref.quantile(.01), 生成1パーセント点=syn.quantile(.01),
                    元99パーセント点=ref.quantile(.99), 生成99パーセント点=syn.quantile(.99),
                    要確認=ks > KS_LIMIT or smd > SMD_LIMIT)
            else:
                row.update(KS=np.nan, SMD=np.nan, 要確認=True)
            numeric_rows.append(row)
    else:
        s = synthetic[column].astype("string").dropna()
        for label, reference in references.items():
            r = reference[column].astype("string").dropna()
            cats = sorted(set(r) | set(s))
            a = r.value_counts(normalize=True).reindex(cats, fill_value=0.)
            b = s.value_counts(normalize=True).reindex(cats, fill_value=0.)
            tv = float((a-b).abs().sum()/2) if len(r) and len(s) else np.nan
            category_rows.append({"比較対象": label, "列名": column, "TV": tv,
                                  "要確認": pd.isna(tv) or tv > TV_LIMIT})
            for cat in cats:
                proportion_rows.append({"比較対象": label, "列名": column, "値": cat,
                    "元割合": a[cat], "生成割合": b[cat], "絶対差": abs(a[cat]-b[cat])})
numeric_report = pd.DataFrame(numeric_rows)
category_report = pd.DataFrame(category_rows)
proportion_report = pd.DataFrame(proportion_rows)
for report, filename in [(numeric_report,"numeric_distribution.csv"), (category_report,"category_distribution.csv"),
                          (proportion_report,"category_proportions.csv")]:
    report.to_csv(OUT / filename, index=False, encoding="utf-8-sig")
if not numeric_report.empty:
    display(numeric_report.sort_values("KS",ascending=False).head(30))
if not category_report.empty:
    display(category_report.sort_values("TV",ascending=False).head(30))

In [ ]:
# 数値同士の関係をSpearman相関で比較（単変量分布だけでは見えない差）。
numeric_cols = [c for c in common if schema.get(c,{}).get("kind") == "numeric"]
corr_rows = []
for label, reference in references.items():
    if len(numeric_cols) < 2:
        break
    a = reference[numeric_cols].sample(n=min(MAX_CORRELATION_ROWS,len(reference)),random_state=SEED).apply(pd.to_numeric,errors="coerce")
    b = synthetic[numeric_cols].sample(n=min(MAX_CORRELATION_ROWS,len(synthetic)),random_state=SEED).apply(pd.to_numeric,errors="coerce")
    ca, cb = a.corr(method="spearman",min_periods=30), b.corr(method="spearman",min_periods=30)
    for i, left in enumerate(numeric_cols):
        for right in numeric_cols[i+1:]:
            av,bv = ca.loc[left,right],cb.loc[left,right]
            diff = abs(av-bv)
            corr_rows.append({"比較対象":label,"列1":left,"列2":right,"元相関":av,"生成相関":bv,
                              "絶対差":diff,"要確認":pd.isna(diff) or diff>CORRELATION_DIFF_LIMIT})
correlation_report = pd.DataFrame(corr_rows)
if not correlation_report.empty:
    display(correlation_report.sort_values("絶対差",ascending=False).head(30))
correlation_report.to_csv(OUT / "numeric_correlations.csv",index=False,encoding="utf-8-sig")

In [ ]:
# 同じ論理チェックを学習・ホールドアウト・生成に適用します。
# 欠損は違反と区別して「比較不能」とします。
def logical_checks(frame):
    result = []
    def add(name, known, bad):
        known = pd.Series(known, index=frame.index).fillna(False).astype(bool)
        bad = pd.Series(bad, index=frame.index).fillna(False).astype(bool) & known
        result.append({"項目":name,"違反件数":int(bad.sum()),"比較可能件数":int(known.sum()),
                       "比較不能件数":int((~known).sum()),
                       "違反率":float(bad.sum()/known.sum()) if known.any() else np.nan})
    for c in ["覚知現着分","現着接触分","接触病着分"]:
        if c in frame:
            v=pd.to_numeric(frame[c],errors="coerce");add(c+" 0～120分",v.notna(),~v.between(0,120))
    for c in frame:
        if c.startswith("覚知") and c.endswith("分") and c!="覚知現着分":
            v=pd.to_numeric(frame[c],errors="coerce")
            low,high=(-120,0) if c=="覚知目撃分" else (-120,120)
            add(c+f" {low}～{high}分",v.notna(),~v.between(low,high))
    if "発生年月日ウツタイン" in frame:
        event=pd.to_datetime(frame["発生年月日ウツタイン"],errors="coerce")
        if "年" in frame:
            y=pd.to_numeric(frame["年"],errors="coerce")
            add("年と発生日の年",event.notna()&y.notna(),event.dt.year.ne(y))
            add("2014年以降",y.notna(),y.lt(2014))
        if "覚知ウツタイン" in frame:
            call=pd.to_datetime(frame["覚知ウツタイン"],errors="coerce")
            add("発生日と覚知日の一致",event.notna()&call.notna(),event.dt.normalize().ne(call.dt.normalize()))
    if {"除細動","施行回数"}.issubset(frame):
        count=pd.to_numeric(frame["施行回数"],errors="coerce")
        add("除細動なし・施行回数1以上",frame["除細動"].notna()&count.notna(),frame["除細動"].eq("なし")&count.ge(1))
    if {"バイスタンダーCPR","心臓マッサージ"}.issubset(frame):
        add("バイスタンダーCPRなし・心臓マッサージあり",frame["バイスタンダーCPR"].notna()&frame["心臓マッサージ"].notna(),
            frame["バイスタンダーCPR"].eq("なし")&frame["心臓マッサージ"].eq("チェックあり"))
    return pd.DataFrame(result)
logic_parts=[]
for label,frame in {**references,"生成":synthetic}.items():
    report=logical_checks(frame)
    if not report.empty:
        report.insert(0,"データ",label);logic_parts.append(report)
logic_report=pd.concat(logic_parts,ignore_index=True) if logic_parts else pd.DataFrame()
display(logic_report)
logic_report.to_csv(OUT / "logical_consistency.csv",index=False,encoding="utf-8-sig")

In [ ]:
# 主要な分布を図に保存。日本語フォント設定に依存しないよう図では内部番号を使用。
plot_columns = [c for c in ["年齢","覚知現着分","現着接触分","接触病着分"]
                if c in common and schema.get(c,{}).get("kind")=="numeric"]
for i,column in enumerate(plot_columns,1):
    a=numeric_values(holdout[column]);b=numeric_values(synthetic[column])
    if a.empty or b.empty:continue
    lo=min(a.quantile(.01),b.quantile(.01));hi=max(a.quantile(.99),b.quantile(.99))
    if hi<=lo:continue
    bins=np.linspace(lo,hi,31)
    fig,ax=plt.subplots(figsize=(8,4))
    ax.hist(a,bins=bins,weights=np.ones(len(a))/len(a),alpha=.5,label="Holdout")
    ax.hist(b,bins=bins,weights=np.ones(len(b))/len(b),alpha=.5,label="Synthetic")
    ax.set(title=f"Feature {i}: central 1-99% range",xlabel="Value",ylabel="Fraction per bin")
    ax.legend();fig.tight_layout();fig.savefig(OUT/f"distribution_{i:02d}.png",dpi=150)
    print(f"Feature {i}: {column}");plt.show();plt.close(fig)

In [ ]:
# 最終概要：型やコードの違反と、分布差の運用目安超過を分けて報告します。
def flagged(report):
    return int(report["要確認"].fillna(True).sum()) if "要確認" in report else 0
summary={"生成ファイル":str(SYNTHETIC_PATH),"モデル":str(CHECKPOINT),"生成件数":len(synthetic),
    "学習件数":len(train),"ホールドアウト件数":len(holdout),
    "モデルSHA256一致":True,
    "構造チェック要確認列数":int((~structure_report["結果"].isin(["適合","定数一致"])).sum()),
    "欠損率差の目安超過数":flagged(missing_report),
    "数値分布差の目安超過数":flagged(numeric_report),
    "カテゴリー分布差の目安超過数":flagged(category_report),
    "相関差の目安超過数":flagged(correlation_report),
    "説明":"分布差の件数は学習・ホールドアウトの比較をそれぞれ計数。生成データは19の後処理済み。臨床的妥当性・プライバシーの合否判定ではない。"}
(OUT/"validation_summary.json").write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding="utf-8")
settings={"missing_diff":MISSING_DIFF_LIMIT,"KS":KS_LIMIT,"SMD":SMD_LIMIT,"TV":TV_LIMIT,
          "correlation_diff":CORRELATION_DIFF_LIMIT,"max_numeric_sample":MAX_NUMERIC_SAMPLE,"seed":SEED}
(OUT/"validation_settings.json").write_text(json.dumps(settings,indent=2),encoding="utf-8")
display(pd.Series(summary,name="結果").to_frame())
print("検証結果の保存先:",OUT)

In [ ]:
# 同じ学習実行の通常・EMA生成結果の欠損率を横並びで保存します。
# 最新出力だけを勝手に選ばず、条件（モデル・seed・件数）ごとに全実行を表示します。
comparison_parts = []
for path in sorted(GENERATED_ROOT.glob("*/provenance.json")):
    try:
        p = json.loads(path.read_text())
        if Path(p.get("source_run", "")).name != SOURCE_RUN.name:
            continue
        stage_path = path.parent / "missingness_stages.csv"
        if not stage_path.is_file():
            continue
        part = pd.read_csv(stage_path)
        part.insert(0,"生成実行",path.parent.name)
        part.insert(1,"種別",p.get("model_type","旧版/不明"))
        part.insert(2,"モデル",Path(p.get("checkpoint", "")).name)
        part.insert(3,"seed",p.get("seed"))
        part.insert(4,"件数",p.get("n_generate"))
        comparison_parts.append(part)
    except (OSError,ValueError):
        continue
if comparison_parts:
    model_comparison = pd.concat(comparison_parts,ignore_index=True)
    model_comparison.to_csv(OUT / "normal_vs_ema_missingness.csv",index=False,encoding="utf-8-sig")
    target_columns = ["除細動","覚知初回心拍再開分","接触病着分"]
    display(model_comparison.loc[model_comparison["列名"].isin(target_columns)])
else:
    print("新版19の欠損段階記録がありません。通常/EMA比較は両モデルの生成後に確認できます。")


## 結果の見方
`model_metadata_checks.csv`：選んだ生成PKLのモデル由来情報を確認。
`schema_checks.csv`：列不足・型・未知カテゴリー・定数を確認。
`missingness_comparison.csv`、`numeric_distribution.csv`、`category_distribution.csv`：単変量の差。
`numeric_correlations.csv`：数値同士の関係の差。欠損が多い・定数のペアは比較不能。
`logical_consistency.csv`：日時・処置の不一致が元データにも存在するか、生成で増えているかを確認。

日付・整数・経過時間は19で後処理されているため、良好な結果がモデル単独の性能を意味するとは限りません。このコードは数値・カテゴリ混合の全ての多変量関係や、個人情報漏洩を網羅する評価ではありません。
実データでの検証は未実行です。模擬データで主要処理を確認しています。